# koilog — журнал силовых тренировок

`koilog` разбирает две выгрузки в одну базу SQLite, считает производные
метрики (1RM, рекорд, скользящее среднее, КОИ) по явным правилам и строит
программу роста нагрузки от последней реальной тренировки — только по тем
весам, которые есть в инвентаре зала.

Вся логика живёт в пакете `koilog` и вызывается из командной строки; этот
ноутбук — обзор поверх той же базы. Он импортирует **те же** функции, что и
CLI, поэтому расхождений между выводом здесь и в терминале нет.

```
python -m koilog load                       # разобрать CSV в sql/training.db
python -m koilog refresh                    # журнал + свежие данные часов в базу
python -m koilog stats                      # сводка по упражнениям за 3 месяца
python -m koilog stats --all                # вся история
python -m koilog plan "Жим гантелей"        # программа + периоды
python -m koilog weeks --exercise "Жим гантелей"   # недельные суммы КОИ
python -m koilog history "Жим гантелей" --from 2026-01-01 --to 2026-10-08
python -m koilog explain "Жим гантелей"     # разбор парсинга и расчёта
python -m koilog check                      # что в данных требует решения
```

Пакет держится на стандартной библиотеке Python 3.13. `matplotlib` и
`ipywidgets` нужны только этому ноутбуку и на CLI не влияют.

## Как читать числа

Короткий словарь, чтобы колонки не приходилось угадывать.

**1RM** — оценка максимального веса на одно повторение по формуле Brzycki
`вес × 36 / (37 − повторы)`. Выше 20 повторов не считается: экстраполяция
перестаёт иметь смысл.

**Для упражнений «тело + жилет»** (`Отжимания`) рабочий вес — это **суммарный**
вес: собственное тело плюс добавка. В таблицах ниже, где нужен вес, он уже
суммарный; в `explain` видно, как из «сырых» 20 кг жилета получается 90 кг
(70 тело + 20 жилет).

| колонка | что значит |
|---|---|
| подх | подходов упражнения в выбранном окне |
| лучший | лучший 1RM **внутри окна** |
| рекорд | лучший 1RM за **всю историю** — база для следующей колонки |
| посл. | 1RM последней тренировки в окне |
| Δ к рекорду | `посл. − рекорд`: 0 = на рекорде, минус = отставание |
| МА30 | скользящее среднее 1RM за **30 дней по этому упражнению** (база КОИ) |
| объём | сумма `вес × повторы` по подходам, в килограммо-повторах |

Раньше на месте «Δ к рекорду» стояла дельта к предыдущей тренировке. Она
честно равнялась нулю на двух одинаковых тренировках подряд (и это выглядело
как поломка), поэтому теперь дельта считается к рекорду, а изменение к прошлой
тренировке печатает `koilog plan` — с датой той тренировки.

**КОИ** — «коэффициент относительной интенсивности», мера усталостной нагрузки.
Формула оригинальная: `КОИ = Σ повторы / (100 − интенсивность%)`, где
интенсивность = `вес / база 1RM`. Делить на 100 обязательно: интенсивность —
доля, а формула выведена в процентах. Выше 98 % от базы КОИ не считается:
при интенсивности около 100 % знаменатель уходит в ноль.

**КОИ считается по каждому упражнению отдельно**, а не по группе: у жима и
тяги в одной группе разные базы 1RM, и общая сумма теряла бы масштаб. Поэтому
и недельные суммы, и суммы за тренировку — по упражнениям.

In [1]:
# Шаг обновления: журнал (sets) + свежие данные часов и производные
# в базу (журнал -> выгрузка -> нагрузка -> эпизоды), ML-блок не трогаем.
# Журнал грузится из data/journal.csv по слепку mtime:size, поэтому свежая
# тренировка появляется сама; отдельный `python -m koilog load` не нужен.
# Печатаем только имя файла и даты: абсолютный путь с именем пользователя
# не должен попадать в сохранённый вывод ноутбука.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

from koilog import etl, stats
from koilog.health import refresh

conn = etl.connect(etl.DEFAULT_DB)
print(etl.DEFAULT_DB.name)
print(refresh.format_freshness(refresh.run(conn)))

training.db
обновление: журнал (файл не менялся - скип), выгрузка (файл не менялся - скип), нагрузка, календарь, эпизоды
выгрузка от 2026-10-21 14:22:18 +0300

таблица           дата      
----------------------------
sets              2026-10-21
aw_daily          2026-10-21
load_daily        2026-10-21
health_days       2026-10-21
ml_features       2026-10-21
ml_score          2026-10-21
illness_episodes  2026-10-11


In [2]:
# Что вообще загрузилось: сколько подходов дал каждый источник и за какой срок.
# То же печатает `python -m koilog load` (блок «источник ... подходов»).
for name, block in etl.summary(conn)["by_source"].items():
    print(f"{name:14} {block['sets']:6} подходов  {block['days']:4} дней  "
          f"{block['first_day']} ... {block['last_day']}")

journal         15451 подходов   945 дней  2018-12-10 ... 2026-10-21
report           5544 подходов   454 дней  2015-06-21 ... 2021-09-17


In [3]:
# Сводка упражнений за окно - то же, что печатает `python -m koilog stats`.
# Окно здесь - три месяца (как у `stats` без ключа --all): упражнения, которыми
# давно не занимались, в неё не попадают.
# Подписи колонок: подх - подходов, лучший/рекорд/посл. - 1RM, Δ к рекорду -
# отставание последней тренировки от рекорда, МА30 - среднее за 30 дней
# по упражнению, объём - вес x повторы.
since = stats.since_for_months(conn, 3)
print(stats.format_report(stats.exercise_report(
    conn, order="best_rm", limit=15, with_rm_only=True, since=since)))
print("лучший 1RM окна, рекорд за всю историю, последний 1RM и отставание "
      "от рекорда; МА30 - база КОИ по упражнению.")

упражнение                     группа                подх  лучший  рекорд   посл. Δ к рекорду    МА30      объём
----------------------------------------------------------------------------------------------------------------
Отжимания                      ОТЖИМАНИЯ               36   111.7   157.3   111.7       -45.5   109.5      20352
Подтягивания на перекладине    ТЯГА ВЕРТИКАЛЬНАЯ       44    98.4   123.3    98.4       -24.9    98.4      20622
Приседания с гантелями         ПРИСЕДАНИЯ              44    79.4    90.7    79.0       -11.7    79.3      17252
Становая тяга на прямых ногах  СТАНОВАЯ ТЯГА           48    79.4    79.4    79.0        -0.5    79.3      18806
Шраги с гантелями              ПЛЕЧИ И ТРАПЕЦИЯ        36    79.4    84.7    79.0        -5.7    79.2      14980
Жим гантелей                   ЖИМ ЛЁЖА                48    74.6    77.8    74.3        -3.4    74.5      18986
Сгибание рук с гантелями       БИЦЕПС                  36    48.0    48.0    48.0         0.0   

In [4]:
# Программа роста нагрузки от последней реальной тренировки: стартовые вес и
# повторы берутся из базы, список весов - из инвентаря зала.
# В шапке видно рекорд, изменение к прошлой тренировке (с её датой) и МА30 -
# базу КОИ. Если в инвентаре нет записанного веса, программа стартует
# с ближайшего не тяжелее и пишет об этом строкой «~».
# Переходы на следующий вес программа больше не печатает - они отдельной
# ячейкой ниже.
print(stats.format_plan(stats.progression_from_db(conn, "Приседания с гантелями")))

=== Приседания с гантелями  (ПРИСЕДАНИЯ) ===
история: 614 подходов, 166 тренировок, 2016-01-27 ... 2026-10-19
лучший 1RM: 68 кг x 10 -> 90.7 (2020-04-13)
рекорд:     90.7 кг за всю историю
последний:  68 кг x 6 -> 79.0 (2026-10-19)
изменение к прошлой тренировке 2026-10-05 (79.4): -0.48 кг (падение)
Δ к рекорду: -11.70 кг (отставание от рекорда -11.70 кг)
МА30 (средний 1RM за 30 дней по упражнению, база КОИ): 79.3
! остановка: 80 кг - верх списка снарядов: дальше рост только повторами на этом весе
инвентарь: гантели в двух руках (вес суммарный), 16 значений, вес делится на 2

программа (база 1RM пересчитывается по ходу):
 шаг  подходы                       база  нагрузка      КОИ  режим
------------------------------------------------------------------
   1  68x6 68x6 68x6                79.0     86.1%     1.30  Load
   2  68x6 68x6 68x7                81.6     83.3%     1.14  Load
   3  68x6 68x7 68x7                81.6     83.3%     1.20  Load
   4  68x7 68x7 68x7                81.

In [5]:
# Переходы на следующий вес: где заканчивается «волна» на текущем снаряде и на
# какой вес программа переходит дальше. Колонка «дельта» - насколько 1RM нового
# веса отличается от текущего (обычно вниз: новый вес берётся с 6 повторов).
# Это отдельная ячейка, а не часть «программы роста»: там те же переходы
# выводились дважды.
plan = stats.progression_from_db(conn, "Жим гантелей")
print(stats.format_transitions(plan.wave))

переходы на следующий вес:
            из  ->  новый вес            1RM   дельта
----------------------------------------------------
64 кг x 8    ->  68 кг x 6         78.97    -0.48
68 кг x 8    ->  72 кг x 6         83.61    -0.80
72 кг x 8    ->  76 кг x 6         88.26    -1.12
76 кг x 8    ->  80 кг x 6         92.90    -1.44



In [6]:
# Недельные суммы КОИ - по КАЖДОМУ упражнению отдельно (оригинальная логика
# КОИ). Колонка «КОИ» - сумма подходов недели по упражнению, «режим» - полоса
# по этой сумме. Отдельной колонки «на подход» нет: полоса задана именно для
# суммы, а усреднение её размывало.
# Полосы недельного КОИ: < 2 лёгкая, 2…3 тяжёлая, 3…4 очень тяжёлая,
# больше 4 «Вы сошли с ума??».
print(stats.format_weekly_koi(stats.weekly_koi(conn, limit=2)))

неделя        упражнение                   группа           подходов      КОИ  режим
------------------------------------------------------------------------------------
2026-W43      Приседания с гантелями       ПРИСЕДАНИЯ              4     1.09  Лёгкий
2026-W43      Становая тяга на прямых нога СТАНОВАЯ ТЯГА           4     1.09  Лёгкий
2026-W43      Жим гантелей                 ЖИМ ЛЁЖА                4     1.06  Лёгкий
2026-W43      Шраги с гантелями            ПЛЕЧИ И ТРАПЕЦИЯ        3     1.06  Лёгкий
2026-W43      Разведение рук стоя          ДЕЛЬТЫ                  4     0.86  Лёгкий
2026-W43      Махи в наклоне одной рукой   ДЕЛЬТЫ                  4     0.85  Лёгкий
2026-W43      Разведение рук с гантелями   СВЕДЕНИЕ                4     0.85  Лёгкий
2026-W41      Разведение рук стоя          ДЕЛЬТЫ                  4     1.44  Лёгкий
2026-W41      Приседания с гантелями       ПРИСЕДАНИЯ              4     1.42  Лёгкий
2026-W41      Становая тяга на прямых нога СТАНОВАЯ ТЯГА

### История упражнения за интервал

Интервал задаётся двумя строками ISO-дат `ГГГГ-ММ-ДД`, **обе границы
включительно**. `None` в любой из них означает «без границы с этой стороны»:
`ИНТЕРВАЛ_ОТ = None` и `ИНТЕРВАЛ_ДО = None` берут всю историю упражнения.

Направление вывода задаёт `ОБРАТНЫЙ_ПОРЯДОК`: `False` — от первой даты к
последней, `True` — от последней к первой (и тогда `limit` оставляет самые
свежие тренировки). Тот же отчёт в терминале:
`python -m koilog history "УПРАЖНЕНИЕ" --from ДАТА --to ДАТА` (флаг `--reverse`
разворачивает порядок).

In [7]:
import datetime

# Меняются четыре строки ниже. Даты - включительно, в ISO (ГГГГ-ММ-ДД).
# None = границы нет: от None - с первой записи, до None - по последнюю.
# ОБРАТНЫЙ_ПОРЯДОК=True покажет те же тренировки от последней даты к первой.
УПРАЖНЕНИЕ = "Жим гантелей"
ИНТЕРВАЛ_ОТ = datetime.date(2026, 1, 1)   # пример: с 1 января 2026
ИНТЕРВАЛ_ДО = None                        # None - по последнюю тренировку
ОБРАТНЫЙ_ПОРЯДОК = False                  # False - от старых; True - от новых


def _iso(value):
    # date -> ISO-строка, None -> None.
    return None if value is None else value.isoformat()


rows = stats.history(
    conn, УПРАЖНЕНИЕ,
    since=_iso(ИНТЕРВАЛ_ОТ), until=_iso(ИНТЕРВАЛ_ДО),
    reverse=ОБРАТНЫЙ_ПОРЯДОК,
)
print(stats.format_history(
    rows, УПРАЖНЕНИЕ,
    since=_iso(ИНТЕРВАЛ_ОТ), until=_iso(ИНТЕРВАЛ_ДО),
    reverse=ОБРАТНЫЙ_ПОРЯДОК,
))
# «верхний подход» - лучший подход тренировки; 1RM - его 1RM; Δ 1RM - к прошлой
# тренировке упражнения; КОИ - сумма подходов тренировки; режим - по KOI_BANDS.

=== Жим гантелей: история за 2026-01-01 … конец (от старых к новым) ===
      дата   верхний подход  подх повторов     объём     1RM   Δ 1RM     КОИ  режим
-----------------------------------------------------------------------------------
2026-01-13        54 кг x 6     4       24      1128    62.7   +0.00    0.96  Optimal
2026-01-20        54 кг x 6     4       24      1212    62.7   +0.00    1.21  Load
2026-01-27        54 кг x 6     4       24      1236    62.7   +0.00    1.50  Load
2026-03-18        54 кг x 6     4       24      1128    62.7   +0.00    1.09  Load
2026-03-24        54 кг x 6     4       24      1128    62.7   +0.00    1.09  Load
2026-03-31        54 кг x 6     4       24      1212    62.7   +0.00    1.36  Load
2026-04-07        54 кг x 6     4       24      1236    62.7   +0.00    1.50  Load
2026-04-14        54 кг x 7     4       25      1290    64.8   +2.09    1.51  Load
2026-04-21        54 кг x 7     4       26      1344    64.8   +0.00    1.53  Load
2026-04-28

In [8]:
# Разбор одного упражнения: сначала - как читаются выгрузки (кодировка,
# разделитель, приоритет дат, правила), затем - от «сырого» значения в файле
# до КОИ, подход за подходом.
# Это ответ на вопрос «откуда взялось 111.7 у Отжиманий»: в колонках видно
# «сырой 20 -> режим vest_only -> свой вес 70 + добавка 20 -> вес 90 -> 1RM».
# Таблицу формирует общая функция stats.format_explain - тот же вид, что в
# `python -m koilog explain "Отжимания"`.
print("=== как читаются выгрузки ===")
for note in etl.parse_notes():
    print(f"  - {note}")

audit = etl.exercise_audit(conn, УПРАЖНЕНИЕ)
if audit is None:
    print(f"\nупражнение «{УПРАЖНЕНИЕ}» в базе не найдено")
else:
    print(f"\n=== разбор «{audit.exercise}» ===")
    print(f"подходов всего: {audit.total}, с 1RM: {audit.with_rm}; "
          f"на времени: {audit.timed}, пропущено: {audit.skipped}, "
          f"выбросов: {audit.outlier}; период {audit.first_day} … {audit.last_day}")
    print("режимы веса: " + ", ".join(f"{n} ({c})" for n, c in audit.weight_modes))

    print("\n=== последние подходы: от сырого значения до КОИ ===")
    print(stats.format_explain(stats.explain_sets(conn, УПРАЖНЕНИЕ, limit=8)),
          end="")

=== как читаются выгрузки ===
  - журнал:  data\journal.csv
  - отчёт:   data\report.csv
  - кодировка: utf-8-sig (BOM снимается), разделитель ячеек ';'
  - даты: журнал - уже ISO (ГГГГ-ММ-ДД); отчёт - «25 сент. 2021 г.» переводится в ISO
  - даты, встречающиеся в обоих файлах: берётся журнал, строки отчёта за этот день пропускаются (журнал подробнее)
  - «ПРОПУЩЕН» и пустые повторы: подход помечается skipped и в метрики не входит
  - разминка: помечается is_warmup и в 1RM/КОИ не входит
  - вес больше 20 кг в упражнении «на своём теле без веса»: читается как секунды (rules.seconds_in_weight), переносится в time_seconds
  - выбросы (rules.check_set): помечаются is_outlier и outlier_rule, остаются в export, но исключаются из 1RM, дельты, МА, КОИ и объёма
  - вес собственного тела оценивается по годам из записей total_bodyweight (см. estimate_bodyweight), константы 70 больше нет

=== разбор «Жим гантелей» ===
подходов всего: 535, с 1RM: 530; на времени: 0, пропущено: 4, выбросов: 0; перио

In [9]:
# Проблемные данные: что требует ручного решения (нераспознанные упражнения,
# подозрительное время, неправдоподобная добавка, вес тела по годам, выбросы).
# То же печатает `python -m koilog check`.
print(etl.quality_report(conn).format(limit=10), end="")

нераспознанные упражнения: 0
подозрительные «повторы» (похоже на секунды): 138
    2015-06-21  Скручивание на коленях в блочном тренажере вес=20.0 повт=None сек=20  упражнение из списка на время
    2015-06-27  Скручивание на коленях в блочном тренажере вес=25.0 повт=None сек=20  упражнение из списка на время
    2015-07-05  Скручивание на коленях в блочном тренажере вес=27.5 повт=None сек=17  упражнение из списка на время
    2015-07-12  Скручивание на коленях в блочном тренажере вес=27.5 повт=None сек=20  упражнение из списка на время
    2015-07-18  Скручивание на коленях в блочном тренажере вес=20.0 повт=None сек=20  упражнение из списка на время
    2015-07-18  Скручивание на коленях в блочном тренажере вес=25.0 повт=None сек=20  упражнение из списка на время
    2015-07-25  Скручивание на коленях в блочном тренажере вес=20.0 повт=None сек=20  упражнение из списка на время
    2015-07-25  Скручивание на коленях в блочном тренажере вес=27.5 повт=None сек=16  упражнение из списка на

## График 1RM

`matplotlib` и `ipywidgets` — единственные нестандартные библиотеки проекта, и
нужны они только ноутбуку: в пакете `koilog` графики остаются ASCII
(`koilog weeks`, `koilog plan`).

```
pip install matplotlib ipywidgets
```

Ячейка ниже рисует **ровно один** график 1RM для выбранного упражнения за
выбранный период. Если `ipywidgets` установлен — выбор упражнения и дат
делается полями; если нет — тем же кодом (`УПРАЖНЕНИЕ`, `СТАРТ`, `ФИНИШ`).

Раньше здесь был баг: каждая смена поля рисовала новый график **поверх**
предыдущего, и в сохранённом выводе их накапливалось по три-четыре. Причина —
`draw_rm` не закрывал фигуру, и pyplot держал все открытые кадры. Теперь
фигура закрывается в `finally`, поэтому интерактивный и статичный варианты
ведут себя одинаково: на экране всегда одна кривая.


In [10]:
# Абсолютный 1RM выбранного упражнения за выбранный период.
#
# Как задать диапазон:
#   * поле «упражнение» - выпадающий список всех упражнений с посчитанным 1RM;
#   * поле «с» и «по» - календарные даты начала и конца, обе включительно.
# Ниже тот же диапазон задаётся кодом (УПРАЖНЕНИЕ/СТАРТ/ФИНИШ): он используется
# как начальные значения виджетов и как единственный способ, если ipywidgets
# не установлен.
#
# Главное правило ячейки: график всегда ОДИН. draw_rm закрывает фигуру через
# plt.close(fig) в finally - без этого pyplot копит открытые кадры, и каждая
# смена поля (или повторный запуск ячейки) добавляла бы ещё одну кривую:
# автор доходил до четырёх графиков в одном выводе.
#
# ipywidgets необязателен: без него ячейка просто рисует один статичный кадр по
# переменным выше - ноутбук не ломается.
#
# Куда развивать дальше: plotly (наведение, зум, экспорт HTML) - если понадобится
# делиться графиком.
import datetime

%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Упражнения, у которых вообще есть 1RM (для выбора в поле).
NAMES = [row.exercise
         for row in stats.exercise_report(conn, with_rm_only=True,
                                          order="exercise")]

# Начальные значения: правьте здесь, они же - старт для виджетов.
УПРАЖНЕНИЕ = "Жим гантелей" if "Жим гантелей" in NAMES else (NAMES or ["—"])[0]
СТАРТ = datetime.date(2015, 1, 1)
ФИНИШ = datetime.date.today()


def draw_rm(exercise, start, end):
    # Нарисовать ровно один график 1RM за интервал [start, end] включительно.
    # Виджет вызывает её при каждом изменении поля, поэтому здесь обязательно
    # закрыть фигуру - иначе кривые складываются друг на друга.
    since = start.isoformat() if start else None
    until = end.isoformat() if end else None
    rows = stats.history(conn, exercise, since=since, until=until)

    fig, ax = plt.subplots(figsize=(11, 4.6))
    try:
        if not rows:
            ax.set_title(f"нет 1RM: «{exercise}» за {since or 'начало'} … "
                         f"{until or 'конец'}")
        else:
            dates = [r.date for r in rows]
            values = [r.rm_session if r.rm_session is not None else float("nan")
                      for r in rows]
            ax.plot(dates, values, marker="o", ms=4, lw=1.6, color="#4c72b0")
            ax.set_title(f"1RM: {exercise} ({dates[0]} … {dates[-1]})")
            ax.set_xlabel("дата")
            ax.set_ylabel("1RM, кг")
            last = values[-1]
            if last == last:  # не NaN
                ax.annotate(f"{last:.1f} кг", xy=(dates[-1], last),
                            xytext=(-8, 10), textcoords="offset points",
                            ha="right", fontweight="bold")
            fig.autofmt_xdate(rotation=30)
        plt.show()
    finally:
        plt.close(fig)


try:
    import ipywidgets as wd
    from IPython.display import display
except ImportError:
    wd = None

if wd is None:
    # Виджетов нет: один статичный кадр по переменным выше.
    draw_rm(УПРАЖНЕНИЕ, СТАРТ, ФИНИШ)
else:
    f_ex = wd.Dropdown(options=NAMES, value=УПРАЖНЕНИЕ,
                       description="упражнение")
    f_from = wd.DatePicker(value=СТАРТ, description="с")
    f_to = wd.DatePicker(value=ФИНИШ, description="по")
    ui = wd.VBox([
        wd.HBox([f_ex]),
        wd.HBox([f_from, f_to]),
    ])
    out = wd.interactive_output(
        draw_rm,
        {"exercise": f_ex, "start": f_from, "end": f_to},
    )
    display(ui, out)


Output()

## О формулах и открытых вопросах

**КОИ по упражнению.** Формула `Σ повторы / (100 − интенсивность%)` применяется
к каждому упражнению отдельно — и за неделю, и за тренировку. Пороги полос
вынесены в `koilog.core`: за неделю `< 2 / 2…3 / 3…4 / > 4`, за одну тренировку
`< 0.4 / 0.4…1 / 1…2 / > 2`. Раньше недельные суммы складывались по группам,
из-за чего жим и тяга делили одну сумму; теперь это исправлено.

**Масштаб КОИ** честный (доли, а не проценты: тренировка в 1,1–1,3 — это
`Load`, а не `Crazy`). Недельные суммы сравниваются по полосам `core.KOI_BANDS_WEEK`,
которые заданы именно для суммы.

**Вес собственного тела** больше не константа 70: он оценивается по годам из
упражнений с суммарным весом (подтягивания, отжимания на брусьях) — сначала
«чистый пол», иначе мода, затем расхождение между упражнениями года
разрешается вниз, потому что жилет добавляет вес. Переопределить:
`python -m koilog load --bodyweight-year 2021=82`.

**76 подходов отчёта за 2018 год** записаны весом жилета вместо суммарного
(`Подтягивания` 25, 31,7, 37,3 кг при теле 67,4). Правила для «слишком мало»
нет — выдумывать порог значит выкидывать настоящие подходы, поэтому строки
помечены в `koilog check` как неправдоподобная добавка, но остаются в базе.